In [84]:
pip install pytorch-forecasting pytorch-lightning torch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import torch
import pytorch_lightning as pl
from pytorch_forecasting import  TimeSeriesDataSet 
from pytorch_forecasting import TemporalFusionTransformer, Baseline, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import SMAPE
from lightning.pytorch import Trainer 
from pytorch_forecasting.metrics import QuantileLoss

In [86]:
import pytorch_forecasting
print(pytorch_forecasting.__version__)

import pytorch_lightning
print(pytorch_lightning.__version__)

1.6.1
2.6.1


In [87]:
df= pd.read_csv("makerere_Cafeteria_synthetic.csv")
df["Date"] = pd.to_datetime(df["Date"])

In [88]:
#sort dataset
df=df.sort_values(["Cafeteria_ID","Date"])

In [89]:
#create time index
df["time_idx"] = df.groupby("Cafeteria_ID").cumcount()

In [90]:
df["month"] = df["Date"].dt.month
df["day_of_week"] = df["Date"].dt.dayofweek
df["week_of_year"] = df["Date"].dt.isocalendar().week
df["is_weekend"] = df["Date"].dt.weekday >= 5
df["is_weekend"] = df["is_weekend"].astype(int)

In [91]:
categorical_cols =[
    "Cafeteria_ID",
    "Cafeteria_Name",
    "Meal",
    "Academic_Period",
    "day_of_week"
]
for col in categorical_cols:
    df[col] = df[col].astype("category")

In [92]:
#define the target variable
target="Portions_Sold"

In [93]:
df.dtypes

Cafeteria_ID                 category
Cafeteria_Name               category
Date                   datetime64[ns]
Day_of_Week                    object
Academic_Period              category
Is_Weekend                       bool
Meal                         category
Portions_Prepared               int64
Portions_Sold                   int64
Waste_Portions                  int64
Waste_Pct                     float64
Price_UGX                      object
Revenue_UGX                    object
Ingredient_Cost_UGX            object
Waste_Cost_UGX                 object
Gross_Profit_UGX               object
Posho_Flour_kg                float64
Beans_kg                      float64
Cooking_Oil_L                 float64
Matooke_kg                    float64
Groundnuts_kg                 float64
Rice_kg                       float64
Chicken_kg                    float64
Offal_kg                      float64
Onions_kg                     float64
Irish_Potatoes_kg             float64
Eggs_units  

In [94]:
df["Revenue_UGX"] = pd.to_numeric(df["Revenue_UGX"].astype(str).str.replace(",", ""))
df["Ingredient_Cost_UGX"] = pd.to_numeric(df["Ingredient_Cost_UGX"].astype(str).str.replace(",", ""))
df["Portions_Sold"] = pd.to_numeric(df["Portions_Sold"])

In [95]:
df["Portions_Sold"] = df["Portions_Sold"].astype(float)

In [96]:
#create the time series dataset
max_encoder_length = 30
max_prediction_length = 7
#create the dataset
training = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="Portions_Sold",
    group_ids=["Cafeteria_ID"],
    
    max_encoder_length=30,
    max_prediction_length=7,
    
    static_categoricals=["Cafeteria_ID","Cafeteria_Name"],
    
    
    time_varying_known_categoricals=[
        "Meal",
        "Academic_Period",
    ],
    
    time_varying_known_reals=[
        "month",
        "day_of_week",
        "week_of_year",
        "is_weekend"
    ],
    time_varying_unknown_reals=[
        "Portions_Sold",
        "Revenue_UGX",
        "Ingredient_Cost_UGX",
        ],
    target_normalizer=GroupNormalizer(
        groups=["Cafeteria_ID"],
        transformation="softplus",
    )
)

In [97]:
training_dataloader = training.to_dataloader(
    train=True, 
    batch_size=64, 
    num_workers=0)

In [98]:
tft= TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=8,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,
    loss=QuantileLoss(),
)

c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [99]:


trainer = Trainer(
    max_epochs=30,
    accelerator="cpu",
    devices=1,
)
trainer.fit(
    tft,
    train_dataloaders=training_dataloader
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │    244 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    112 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │    154 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  4.8 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │  2.7 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    610 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │    119 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 21.7 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 21.7 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 358                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\loops\training_epoch_loop.py:500: ReduceLROnPlateau 
conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.

`Trainer.fit` stopped: `max_epochs=30` reached.


In [100]:
predictions = tft.predict(training_dataloader)

c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:485: Your `predict_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
c:\Users\USER\anacondaApp\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
